# Đánh giá thăm dò Development 7 ngày — nhiều seed

Chạy ba seed cố định `(7, 11, 19)` trên cùng input UrbanEV SZH 7 ngày. Mỗi seed huấn luyện Source một lượt qua 65.888 mẫu, sau đó chạy target `1001` (6 mẫu ban đầu, một lượt) và `1871` (`source_checkpoint_only`, 0 bước target), mỗi target replay đủ 90 origin. Hai nhóm được báo cáo riêng. Đây là thăm dò Development, không phải ma trận baseline/ablation đầy đủ, cấu hình khóa hay đánh giá Final.

Chọn runtime T4 trước khi Run all. CLI ở commit ghim chỉ chạy CPU; T4 được giữ làm loại runtime theo yêu cầu, nhưng GPU không được sử dụng cho mô hình. Notebook không chọn seed hoặc tham số theo kết quả. Mỗi Run all tạo RUN_ID mới và thư mục Drive mới; không ghi đè artifact cũ.


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CODE_SHA = "85e45606789eaff7e2e929eb7a7ee353a4ee49f9"
INPUT_FILE_ID = "17pfYqnnSq0Uao2zDNdb8h7GAYfotu_w4"
DRIVE_RUN_PARENT = Path("/content/drive/MyDrive/hung-sprint2-runs")
RUN_ID = "hung-urbanev-7d-dev-multiseed-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SEEDS = (7, 11, 19)
TARGETS = ("1001", "1871")
EVAL = {
    "source_steps": 2059,
    "source_batch_size": 32,
    "target_steps": 3,
    "target_batch_size": 2,
    "learning_rate": 0.001,
    "retention_weight": 0.1,
    "epsilon": 1e-6,
    "max_origins": 90,
    "level_one": 0.1,
    "level_two": 0.2,
    "initial_level": 0.0,
    "block_days": 14,
    "min_complete_days": 5,
    "delta": 0.0,
    "alpha": 0.05,
    "bootstrap_segment_days": 2,
    "bootstrap_resamples": 100,
    "challenger_rule": "bonferroni_two",
}
print("ma_luot_chay", RUN_ID)

In [ ]:
from importlib.metadata import version

repo = Path("/content/hung-sprint2-code-" + RUN_ID)
code_bundle = Path("/content/drive/MyDrive/colab-code-85e4560-branch-20261023.bundle")
expected_code_bundle_sha = "08120b8eb6b63db6eb78518d4f333cacc8f402ff2d9824bee400d60016e4a0b2"
if repo.exists():
    raise FileExistsError(repo)
if not code_bundle.is_file():
    raise FileNotFoundError(code_bundle)
actual_bundle_sha = hashlib.sha256(code_bundle.read_bytes()).hexdigest()
if actual_bundle_sha != expected_code_bundle_sha:
    raise ValueError("Code bundle SHA-256 mismatch")
subprocess.run(["git", "clone", str(code_bundle), str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", CODE_SHA], check=True)
actual_sha = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
if actual_sha != CODE_SHA:
    raise ValueError("Checked-out code SHA mismatch")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements-dev.txt")], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
        "-r",
        str(repo / "requirements-model.txt"),
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "gdown==5.2.0"], check=True)
if version("torch") != "2.8.0+cpu":
    raise RuntimeError(f"Unexpected torch version: {version('torch')}")
if version("gdown") != "5.2.0":
    raise RuntimeError(f"Unexpected gdown version: {version('gdown')}")
pip_audit = subprocess.run(
    [sys.executable, "-m", "pip", "check"], text=True, capture_output=True
)
if pip_audit.returncode:
    print("canh_bao_goi_colab:", pip_audit.stdout.strip(), pip_audit.stderr.strip())
print("commit_code", actual_sha, "sha256_archive_code", actual_bundle_sha)


def run_cli(*args):
    command = [sys.executable, "-m", "experiments.hung_sprint2.cli", *map(str, args)]
    print("lenh_chay:", " ".join(command))
    subprocess.run(command, cwd=repo, check=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_drive_run(folder):
    folder = Path(folder)
    marker = folder / "handoff_complete.json"
    if not marker.is_file():
        raise ValueError(f"Drive handoff is incomplete: {folder}")
    recorded = json.loads(marker.read_text(encoding="utf-8"))
    if (
        recorded.get("schema") != "hung-sprint2-drive-handoff-v1"
        or recorded.get("code_sha") != CODE_SHA
    ):
        raise ValueError("Unexpected Drive handoff schema/code")
    actual = {
        p.relative_to(folder).as_posix() for p in folder.rglob("*") if p.is_file() and p != marker
    }
    if actual != set(recorded["files"]):
        raise ValueError("Drive handoff file set differs")
    for name, expected in recorded["files"].items():
        if sha256_file(folder / name) != expected:
            raise ValueError(f"Drive handoff hash mismatch: {name}")
    return recorded


def publish_run(local_folder, drive_folder):
    local_folder = Path(local_folder)
    drive_folder = Path(drive_folder)
    drive_folder.parent.mkdir(parents=True, exist_ok=True)
    if drive_folder.exists():
        raise FileExistsError(drive_folder)
    shutil.copytree(local_folder, drive_folder)
    files = {}
    for original in local_folder.rglob("*"):
        if original.is_file():
            name = original.relative_to(local_folder).as_posix()
            digest = sha256_file(original)
            if digest != sha256_file(drive_folder / name):
                raise ValueError(f"Drive copy mismatch: {name}")
            files[name] = digest
    marker = {
        "schema": "hung-sprint2-drive-handoff-v1",
        "code_sha": CODE_SHA,
        "files": files,
    }
    with (drive_folder / "handoff_complete.json").open("x", encoding="utf-8") as stream:
        json.dump(marker, stream, indent=2, sort_keys=True)
        stream.write("\n")
    verify_drive_run(drive_folder)
    return drive_folder

In [ ]:
archive = Path("/content/inputs-urbanev-SZH-v1.zip")
input_root = Path("/content/hung-sprint2-data") / RUN_ID
if input_root.exists():
    raise FileExistsError(input_root)
if not archive.exists():
    import gdown
    downloaded = gdown.download(id=INPUT_FILE_ID, output=str(archive), quiet=False)
    if downloaded != str(archive):
        raise RuntimeError("Input download failed")
subprocess.run(
    [
        sys.executable,
        "-m",
        "experiments.hung_sprint2.cli",
        "verify-input-zip",
        "--archive",
        str(archive),
        "--package",
        archive.name,
    ],
    cwd=repo,
    check=True,
)
with zipfile.ZipFile(archive) as package:
    names = package.namelist()
    prefix = "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-"
    if len(names) != len(set(names)) or any(
        not name.startswith(prefix) or Path(name).is_absolute() or ".." in Path(name).parts
        for name in names
    ):
        raise ValueError("Unexpected archive entry")
    input_root.mkdir(parents=True)
    package.extractall(input_root)
run = input_root / "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-7d"
manifest_sha = hashlib.sha256((run / "manifest.json").read_bytes()).hexdigest()
print("bo_dau_vao", run.name, "sha256_manifest", manifest_sha)

In [ ]:
run_root = Path("/content/hung-sprint2-runs") / RUN_ID
run_root.mkdir(parents=True, exist_ok=False)
results = []
for seed in SEEDS:
    seed_root = run_root / f"seed-{seed}"
    source_out = seed_root / "source"
    run_cli(
        "source-smoke", "--input-run", run, "--manifest-sha256", manifest_sha,
        "--output", source_out, "--steps", EVAL["source_steps"],
        "--batch-size", EVAL["source_batch_size"], "--seed", seed,
        "--learning-rate", EVAL["learning_rate"], "--calendar-hidden", 32,
    )
    source_record = json.loads((source_out / "run_manifest.json").read_text())
    assert source_record["status"] == "completed"
    assert source_record["source_examples_used"] == source_record["source_examples_available"] == 65888
    for entity_id in TARGETS:
        entity_root = seed_root / f"entity-{entity_id}"
        target_out = entity_root / "target"
        replay_out = entity_root / "replay-90"
        run_cli(
            "target-smoke", "--input-run", run, "--manifest-sha256", manifest_sha,
            "--source-input-run", run, "--source-manifest-sha256", manifest_sha,
            "--source-checkpoint", source_out / "source_checkpoint.pt",
            "--entity-id", entity_id, "--output", target_out,
            "--steps", EVAL["target_steps"], "--batch-size", EVAL["target_batch_size"],
            "--seed", seed, "--learning-rate", EVAL["learning_rate"],
            "--retention-weight", EVAL["retention_weight"], "--epsilon", EVAL["epsilon"],
        )
        target_record = json.loads((target_out / "run_manifest.json").read_text())
        if entity_id == "1001":
            assert target_record["initialization"] == "target_initial_adaptation"
            assert target_record["target_examples_available"] == 6 and target_record["steps"] == 3
        else:
            assert target_record["initialization"] == "source_checkpoint_only"
            assert target_record["target_examples_available"] == 0 and target_record["steps"] == 0
        run_cli(
            "replay-development", "--input-run", run, "--manifest-sha256", manifest_sha,
            "--source-input-run", run, "--source-manifest-sha256", manifest_sha,
            "--source-checkpoint", source_out / "source_checkpoint.pt",
            "--target-checkpoint", target_out / "target_checkpoint.pt",
            "--entity-id", entity_id, "--output", replay_out,
            "--max-origins", EVAL["max_origins"], "--block-days", EVAL["block_days"],
            "--min-complete-days", EVAL["min_complete_days"],
            "--bootstrap-segment-days", EVAL["bootstrap_segment_days"],
            "--bootstrap-resamples", EVAL["bootstrap_resamples"], "--seed", seed,
            "--level-one", EVAL["level_one"], "--level-two", EVAL["level_two"],
            "--initial-level", EVAL["initial_level"], "--delta", EVAL["delta"],
            "--alpha", EVAL["alpha"], "--epsilon", EVAL["epsilon"],
            "--challenger-rule", EVAL["challenger_rule"],
        )
        replay_record = json.loads((replay_out / "run_manifest.json").read_text())
        assert replay_record["status"] == "completed" and replay_record["runner_cursor"] == 90
        assert replay_record["coverage"]["scored"] > 0
        results.append({
            "seed": seed, "entity_id": entity_id,
            "stratum": target_record["initialization"],
            "source_checkpoint_sha256": source_record["checkpoint_sha256"],
            "target_checkpoint_sha256": target_record["checkpoint_sha256"],
            "runner_checkpoint_sha256": replay_record["final_runner_checkpoint_sha256"],
            "coverage": replay_record["coverage"],
            "metrics_kwh": replay_record["metrics_kwh"],
            "duration_seconds": {
                "source": source_record["duration_seconds"],
                "target": target_record["duration_seconds"],
                "replay": replay_record["duration_seconds"],
            },
        })
        print("Development", seed, entity_id, replay_record["coverage"], replay_record["metrics_kwh"]["live"])
with (run_root / "development_summary.json").open("x", encoding="utf-8") as stream:
    json.dump({"run_id": RUN_ID, "code_sha": CODE_SHA, "input_manifest_sha256": manifest_sha,
               "config": EVAL, "results": results}, stream, ensure_ascii=False, indent=2)
print("hoan_tat_development", len(results), "luot")


In [ ]:
# Chỉ bàn giao sau khi so hash các file đã sao chép; không ghi đè Drive run cũ.
assert len(results) == len(SEEDS) * len(TARGETS)
drive_run = publish_run(run_root, DRIVE_RUN_PARENT / RUN_ID)
print("thu_muc_drive", drive_run)
print("so_luot_development", len(results))
